# 03 — CATE with Meta-Learners (T / S / X) — Day 13

Causal Inference for Marketing Attribution — **Week 2, Stage 2**.

**Labelling convention (AGENTS.md §2):** everything below is **estimated
(simulated)** — CATE estimates computed on the clearly-labelled `sim_*`
marketing layer. None of these numbers is an observed Olist fact. The
unobserved confounder `sim_u` is **never a feature** of any learner.

**What this notebook shows:** do the incremental effects vary with observed
customer features? Three meta-learners (T / S / X from causalml 0.17, all
HistGradientBoosting regressors on the probability/revenue scale) are fitted
per channel × outcome, their CATE curves are compared, and a **learner
agreement map** is rendered as the Day-13 validation hook.


In [1]:
# --- setup ---------------------------------------------------------------
import sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd

from src.config import load_config, project_path
cfg = load_config()

TABLES = project_path(cfg["paths"]["results"], cfg["results"]["tables"])
FIGURES = project_path(cfg["paths"]["results"], cfg["results"]["figures"])

cate = pd.read_parquet(TABLES / "cate_estimates.parquet")
summary = pd.read_csv(TABLES / "cate_summary.csv")
agreement_detail = pd.read_csv(TABLES / "cate_agreement.csv")

print(f"unit-level CATE rows: {len(cate):,}")
print("summary rows:", len(summary), "| agreement detail rows:", len(agreement_detail))


unit-level CATE rows: 759,864
summary rows: 24 | agreement detail rows: 24


## 1. Method & identification (stated before fitting)

| Learner | How it estimates τ(x) | Known behaviour |
|---|---|---|
| **T** | two separate outcome regressions (treated / control arm) | higher variance in low-support cells |
| **S** | one outcome regression, treatment as a feature | shrinks effects toward a constant |
| **X** | outcome + mirrored effect regressions, uses propensity p(X) | tightest τ structure |

- Features = the Day-4 **adjustment set** per channel (email: order_count,
  recency_days, review_score_avg, total_revenue; social/search: order_count,
  tenure_days, total_revenue; display: recency_days, total_revenue).
- **Why regressors on the 0/1 outcome:** causalml's classifier path calls the
  base learner's hard `predict` (class labels), collapsing the CATE to ~0.
  Regressors on 0/1 yield a valid probability-scale CATE (Day-13 finding).
- Assumptions: exchangeability given X with **sim_u unobserved** (CATE inherits
  the Days 8–12 ATE bias); positivity/overlap per Day 6 (email weak overlap
  handled there); consistency + SUTVA as the whole project; in-sample CATE
  (no cross-fitting — deferred to Day 18).


In [2]:
summary[["channel", "outcome_label", "learner_label",
          "mean_cate", "ci_lower", "ci_upper", "sd_cate"]].assign(
    ci=lambda d: d.apply(lambda r: f"[{r.ci_lower:.4f}, {r.ci_upper:.4f}]", axis=1)
).drop(columns=["ci_lower", "ci_upper"]).style.format({"mean_cate": "{:.4f}", "sd_cate": "{:.4f}"})


,channel,outcome_label,learner_label,mean_cate,sd_cate,ci
0,email,14-day conversion,T-learner,0.1273,0.0139,"[0.1272, 0.1273]"
1,email,14-day conversion,S-learner,0.1230,0.0136,"[0.1229, 0.1231]"
2,email,14-day conversion,X-learner,0.1271,0.0082,"[0.1270, 0.1271]"
3,email,14-day revenue (R$),T-learner,17.2720,3.8341,"[17.2452, 17.2959]"
4,email,14-day revenue (R$),S-learner,16.3555,3.1348,"[16.3339, 16.3740]"
5,email,14-day revenue (R$),X-learner,17.2384,3.2269,"[17.2191, 17.2606]"
6,social,14-day conversion,T-learner,0.1097,0.0126,"[0.1096, 0.1098]"
7,social,14-day conversion,S-learner,0.1015,0.0093,"[0.1014, 0.1016]"
8,social,14-day conversion,X-learner,0.1082,0.0064,"[0.1082, 0.1083]"
9,social,14-day revenue (R$),T-learner,16.6195,3.1555,"[16.6002, 16.6395]"


## 2. Learner agreement map (validation hook)

Each subplot shows, per channel × outcome, the mean CATE by **baseline-risk
decile** for each learner (T red / S green / X blue). Three overlaid curves
that track each other = the learners agree on the *shape* of heterogeneity.

**Gate (config-calibrated):** every learner's mean CATE within the Day-12 OLS
ATE tolerance, AND max pairwise decile-curve spread ≤ 25% of the effect size.
Per-unit (individual) rank agreement is reported but deliberately **not**
gated — on this DGP the observed-X heterogeneity signal is tiny, so per-unit
CATE ordering is weak by construction.


In [3]:
from src.causal.cate import learner_agreement
agreement = learner_agreement(cate, cfg)
agreement["status"][["channel", "outcome_label", "max_abs_mean_align",
                     "max_decile_rel", "decile_tolerance", "status"]]


C:\Users\DAX\Desktop\projects\TOP3 DATA SCIENCE\Causal Inference for Marketing Attribution\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,channel,outcome_label,max_abs_mean_align,max_decile_rel,decile_tolerance,status
0,email,14-day conversion,0.004399,0.040552,0.25,OK
1,email,14-day revenue (R$),0.755747,0.060948,0.25,OK
2,social,14-day conversion,0.004654,0.077093,0.25,OK
3,social,14-day revenue (R$),0.890778,0.091667,0.25,OK
4,search,14-day conversion,0.005275,0.039436,0.25,OK
5,search,14-day revenue (R$),0.975243,0.052783,0.25,OK
6,display,14-day conversion,0.003760,0.052524,0.25,OK
7,display,14-day revenue (R$),0.897407,0.068942,0.25,OK


In [4]:
from src.causal.cate import render_agreement_map, render_cate_by_channel

fig, map_path = render_agreement_map(cate, agreement, cfg, FIGURES)
fig.show()
print("saved ->", map_path)


saved -> C:\Users\DAX\Desktop\projects\TOP3 DATA SCIENCE\Causal Inference for Marketing Attribution\results\figures\learner_agreement_map.html


### 2.1 The honest reading of the map

- **Aggregate level — agreement is real.** Mean CATEs match the Day-12 ATE and
  the decile curves track each other (max spread ≤
  `agreement["status"]["max_decile_rel"].max()` of the effect size). The signal
  that survives is smooth: the effect is nearly flat, mildly tilted by baseline
  risk. That is what a constant-log-odds embedded effect looks like on the
  probability scale.
- **Individual level — agreement is weak by design.** Pairwise rank Spearman
  sits ~0.3–0.8: personalized CATE ordering on observed X is **not reliable**.
  `sim_u` dominates the assignment, so the little heterogeneity these learners
  can see is drowned by their own bias/variance structure.
- **Agreement is not truth.** Display (simulated ground-truth effect 0.00) again
  shows a CATE ≈ +0.11 pp — an apparent effect driven by unobserved selection,
  not by treatment. Learner agreement does not certify the effect LEVEL.


In [5]:
import plotly.graph_objects as go

# bar summary with bootstrap CI per learner
fig2, bar_path = render_cate_by_channel(summary, cfg, FIGURES)
fig2.show()
print("saved ->", bar_path)


saved -> C:\Users\DAX\Desktop\projects\TOP3 DATA SCIENCE\Causal Inference for Marketing Attribution\results\figures\cate_by_channel.html


## 3. What this means for uplift (Day 14)

The CATE curves being essentially flat on observed X is a *finding*, not a
failure: it says observed features carry little heterogeneity, so uplift
modelling (Day 14) must report persuadable segments and Qini curves honestly at
the aggregate level and be transparent that `sim_u`-driven personalization is
invisible to every model built on this data.


In [6]:
# required-token sanity: the report carrying the full agreement story
report = (project_path(cfg["paths"]["reports"]) / "cate_effects.md").read_text(encoding="utf-8")
for token in ["estimated (simulated)", "Learner agreement map", "sim_u", "Limitations"]:
    assert token in report, token
print("report OK, tokens present")


report OK, tokens present
